# The rebuilt gold sheets on explore200: a preview

A preview and sanity check of `README.md`'s conditions B (video + gold sheet, csv) and C (video + gold sheet +
closest-point relations) on explore200's 200 questions
(`experiments/2026-11-18-scene-sheet-formats/explore200_ids.txt`, 25 scenes), with the gold sheets rebuilt at
`87f3685` (`build_gold_sheets.py`, `tasks/openvocab_adapter/out_full`, checked against the README's hashes). It does not
change the conditions or the plan, and its numbers are not the experiment's results.

- **Settings** as the README's runs and the structured-CoT pilot: Qwen3.5-2B, thinking off, 16 frames `uniform128`,
  `--max-new-tokens 2048 --final-answer`, up to 8 questions of one video per batch.
- **A (video only) is not run again**: it is the pilot's `A_video.jsonl`
  (`results/zeroshot/vsibench/structured_cot/explore200/`, in the code zip). The notebook checks that the eval shows
  every question the frames A was run on, and reruns A on one scene to check that the inputs (and, normally, the
  answers) are the same.
- **Pilot against rebuild:** the summary sets the new B and C beside the pilot's `gold_csv` and `gold_relations`, task by
  task, and says whether they are the same answer for answer. The pilot's answers copy the rebuilt sheets' numbers
  (its closest-point distances and box sizes, where these differ from the older 75-scene sheets of
  `data/scene_sheets_75`), so the same answers throughout are a possible outcome, not a failure.

About 25 minutes on an L4: installing about 5, the 25 videos and the sheets about 5, the check on one scene about 3
(loading the model), and B and C about 5 each.

**After a crash**, run the notebook again from the top: B and C continue from their last saved batch on Drive.

**File on Google Drive**, in `MyDrive/mvp/`: `mvp-3d-vlm-preview.zip`, the code at `87f3685`, made in the repository
with

```
git archive --format=zip -o mvp-3d-vlm-preview.zip 87f3685 src experiments/2026-11-18-scene-sheet-formats experiments/2026-11-19-gold-sheet-relations results/zeroshot/vsibench/structured_cot/explore200 requirements.txt pyproject.toml
```

VSI-Bench's `test.jsonl`, its metadata and the 25 videos are downloaded here. Predictions go to
`MyDrive/mvp/gold_preview_results/`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-preview.zip"
RESULTS_DIR = f"{DRIVE_DIR}/gold_preview_results"
CODE_COMMIT = "87f36854838a3af95dc385bbe703e95ad036fd98"
WORK = "/content/mvp-3d-vlm"
EXP = f"{WORK}/experiments/2026-11-19-gold-sheet-relations"
IDS_PATH = f"{WORK}/experiments/2026-11-18-scene-sheet-formats/explore200_ids.txt"
PILOT = f"{WORK}/results/zeroshot/vsibench/structured_cot/explore200"  # A_video, and the pilot's gold_csv / gold_relations
SHEETS_ROOT = f"{WORK}/tasks/openvocab_adapter/out_full"
VSI_REPO, VSI_REVISION = "nyu-visionx/VSI-Bench", "3a6a1cf9ac54c403509b15260897cc6ddc41b408"
VSI_SHA256 = "aa3ff8cc2010e555f4f865ff96483812ec0b275005b68562f53bb114ad4db9db"
INDEX_SHA256 = "22e0ee1ed3400a5ca59bc49c3791f433265115b4301c183b9772e4e014f5573c"    # README, Runs step 1
OBJECTS_SHA256 = "bc0214c926be913606031474df123b2d6ee749fba39f799d56f4e8b1b21f41a5"
MODEL = "Qwen/Qwen3.5-2B"
COMMON = ["--model", MODEL, "--nframes", "16", "--sampling", "uniform128", "--max-new-tokens", "2048",
          "--final-answer", "--batch-size", "8", "--token-budget", "120000"]
CONDITIONS = {"B_gold_csv": "gold", "C_gold_relations_csv": "gold_relations"}  # name -> --sheet-source
OLD = {"B_gold_csv": "gold_csv", "C_gold_relations_csv": "gold_relations"}       # the pilot's files, old sheets

import hashlib, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
shutil.rmtree(WORK, ignore_errors=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(WORK)
    commit = z.comment.decode()  # git archive writes the commit it archived as the zip's comment
assert commit == CODE_COMMIT, f"{CODE_ZIP} is commit {commit or 'unknown'}, not {CODE_COMMIT}: make it as the first cell says"
os.makedirs(RESULTS_DIR, exist_ok=True)

# requirements.txt pins torch for the cluster; Colab's own torch is built for its driver and is kept
reqs = [l.split("#")[0].strip() for l in open(f"{WORK}/requirements.txt").read().splitlines()]
reqs = [r for r in reqs if r and not r.startswith("--extra-index-url") and r.split("==")[0] not in ("torch", "torchvision")]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r for r in reqs if r != "decord"]], check=True)
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"]).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eva-decord"], check=True)  # same module, newer Pythons
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", WORK], check=True)
sys.path.insert(0, f"{WORK}/src")
ENV = {**os.environ, "PYTHONPATH": f"{WORK}/src"}

import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > GPU (L4 or A100)"
GPU = torch.cuda.get_device_name(0)
print(f"code {commit}; torch {torch.__version__}; GPU {GPU}")

## Data and sheets

`test.jsonl` at the pinned revision, the gold sheets of all 288 scenes rebuilt exactly as the README's step 1 (about a
minute on the CPU; it must give the README's two hashes), `check_sheets.py` on the 200 questions, the 25 videos, and
each video's 128 candidate frames (`frame_indices.npy`) as the VGGT export takes them.

In [ ]:
import json
import numpy as np
from huggingface_hub import HfFileSystem, hf_hub_download

RAW, VGGT = Path(WORK) / "data/raw/VSI-Bench", Path(WORK) / "data/vggt"
RAW.mkdir(parents=True, exist_ok=True)
shutil.copy(hf_hub_download(VSI_REPO, "test.jsonl", repo_type="dataset", revision=VSI_REVISION), RAW / "test.jsonl")
assert hashlib.sha256((RAW / "test.jsonl").read_bytes()).hexdigest() == VSI_SHA256, "not the expected test.jsonl"

def run(cmd):
    r = subprocess.run(cmd, cwd=WORK, env=ENV, capture_output=True, text=True)
    print(r.stdout[-2500:])
    assert r.returncode == 0, r.stderr[-3000:]
    return r.stdout

built = run([sys.executable, f"{EXP}/build_gold_sheets.py", "--questions", str(RAW / "test.jsonl"),
             "--prompts", f"{EXP}/prompts_full.jsonl", "--out", SHEETS_ROOT])
assert f"index.csv sha256 {INDEX_SHA256}" in built and f"objects.csv sha256 {OBJECTS_SHA256}" in built, \
    "the rebuilt sheets are not the README's"
run([sys.executable, f"{EXP}/check_sheets.py", "--sheets-root", SHEETS_ROOT, "--questions", str(RAW / "test.jsonl"),
     "--ids", IDS_PATH])
print("OK: the sheets are the README's, and every question renders in B and C")

In [ ]:
IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]
QUESTIONS = {int(q["id"]): q for q in map(json.loads, open(RAW / "test.jsonl"))}
SCENES = sorted({f"{QUESTIONS[q]['dataset']}/{QUESTIONS[q]['scene_name']}" for q in IDS})

fs = HfFileSystem()
for dataset in sorted({s.split("/")[0] for s in SCENES}):
    todo = [s for s in SCENES if s.startswith(dataset + "/") and not (RAW / f"{s}.mp4").exists()]
    if todo:  # read out of the per-dataset zip without downloading it whole
        with fs.open(f"datasets/{VSI_REPO}@{VSI_REVISION}/{dataset}.zip", "rb", block_size=2**20) as f, zipfile.ZipFile(f) as z:
            for s in todo:
                (RAW / s).parent.mkdir(parents=True, exist_ok=True)
                (RAW / f"{s}.mp4").write_bytes(z.read(f"{s}.mp4"))
print(f"{len(SCENES)} videos")

# frame_indices.npy as mvp.reconstruction.extract.load_frames takes it: min(128, N) frames by np.linspace(0, N - 1)
from decord import VideoReader, cpu
for s in SCENES:
    n = len(VideoReader(str(RAW / f"{s}.mp4"), ctx=cpu(0)))
    (VGGT / s).mkdir(parents=True, exist_ok=True)
    np.save(VGGT / s / "frame_indices.npy", np.linspace(0, n - 1, num=min(128, n), dtype=int).astype(np.int32))

# Check 1: the 16 frames the eval will show (its own frame_indices) are the frames A_video was run on
from mvp.eval.vsibench import frame_indices
A = {r["id"]: r for r in map(json.loads, open(f"{PILOT}/A_video.jsonl"))}
assert sorted(A) == sorted(IDS), "A_video.jsonl does not answer explore200's 200 questions"
bad = [q for q in IDS if frame_indices(f"{RAW}/{QUESTIONS[q]['dataset']}/{QUESTIONS[q]['scene_name']}.mp4",
                                       "uniform128", 16, VGGT) != A[q]["frame_indices"]]
assert not bad, f"the eval would show other frames than A_video's for questions {bad[:10]}"
print("OK: for every question the eval shows the 16 frames A_video was run on")

## A on one scene

A_video is reused, so its inputs must be the ones this code builds. Check 2: A is run again on the first scene's
questions (one batch); the input lengths and frames must equal A_video's. The answers normally do too (greedy
decoding, the same batch); if some differ, it is printed, not stopped on.

In [ ]:
first = SCENES[0].split("/")[1]
check_ids = [q for q in IDS if QUESTIONS[q]["scene_name"] == first]
Path("/content/a_check_ids.txt").write_text("\n".join(map(str, check_ids)) + "\n")
Path("/content/a_check.jsonl").unlink(missing_ok=True)
run([sys.executable, "-m", "mvp.eval.vsibench", *COMMON, "--ids", "/content/a_check_ids.txt", "--out", "/content/a_check.jsonl"])
again = {r["id"]: r for r in map(json.loads, open("/content/a_check.jsonl"))}
assert sorted(again) == sorted(check_ids)
differ = [q for q in check_ids if (again[q]["n_input_tokens"], again[q]["frame_indices"]) != (A[q]["n_input_tokens"], A[q]["frame_indices"])]
assert not differ, f"A's inputs differ from A_video's for questions {differ}: A_video cannot be reused"
answers = [q for q in check_ids if again[q]["prediction"] != A[q]["prediction"]]
print(f"OK: {len(check_ids)} questions of {SCENES[0]}: the same inputs as A_video; "
      + (f"different answers for {answers}" if answers else "the same answers"))

## B and C

Each continues from its file on Drive only if this notebook started that file (`<name>.started` holds the commit,
the sheets' hash and the source); a file from anywhere else stops the notebook, since the eval would take its answers
as done and skip them.

In [ ]:
import time
from huggingface_hub import snapshot_download

for name, source in CONDITIONS.items():
    out, marker = Path(f"{RESULTS_DIR}/{name}.jsonl"), Path(f"{RESULTS_DIR}/{name}.started")
    stamp = f"{CODE_COMMIT} {INDEX_SHA256} {source}\n"
    if out.exists() and not (marker.exists() and marker.read_text() == stamp):
        raise RuntimeError(f"{out} was not started by this notebook with these sheets, and the eval would skip every "
                           "question already in it: delete it (or move it out of RESULTS_DIR) and run this cell again")
    marker.write_text(stamp)
    t0 = time.time()
    run([sys.executable, "-m", "mvp.eval.vsibench", *COMMON, "--sheet-source", source, "--sheet-format", "csv",
         "--sheets-root", SHEETS_ROOT, "--ids", IDS_PATH, "--out", f"{RESULTS_DIR}/{name}.jsonl"])
    print(f"{name}: {(time.time() - t0) / 60:.1f} min on {GPU}")
Path(f"{RESULTS_DIR}/model_revision.txt").write_text(
    f"model: {MODEL}\nrevision: {Path(snapshot_download(MODEL, local_files_only=True)).name}\ncode commit: {CODE_COMMIT}\n"
    f"gpu: {GPU}\n")

## Summary

Per task, ×100, with the eval's own `summarize`: A (the pilot's), B and C from the pilot and from
this run, and how many answers changed. Check 3: B and C answer all 200 questions, saw
the same frames as A, and their prompts are longer (the sheet is in them). A file that was in `RESULTS_DIR` before
this notebook started it never gets here (B and C cell). The numbers are a preview, not the experiment's results.

In [ ]:
import pandas as pd
from mvp.benchmarks.vsibench import summarize

def load(path):
    df = pd.read_json(path, lines=True)
    return df[df["id"].isin(IDS)].set_index("id").sort_index()

runs = {"A": load(f"{PILOT}/A_video.jsonl")}
for name, old in OLD.items():
    runs[f"{name[0]} pilot"] = load(f"{PILOT}/{old}.jsonl")
    runs[f"{name[0]} new"] = load(f"{RESULTS_DIR}/{name}.jsonl")
problems = []
for c in ("B new", "C new"):
    if len(runs[c]) != len(IDS):
        problems.append(f"{c}: {len(runs[c])} of {len(IDS)} questions")
    elif any(f != a for f, a in zip(runs[c]["frame_indices"], runs["A"]["frame_indices"])):
        problems.append(f"{c}: other frames than A's")
    elif (runs[c]["n_input_tokens"] <= runs["A"]["n_input_tokens"]).any():
        problems.append(f"{c}: some prompts are not longer than A's")
assert not problems, "\n".join(problems)
for c in ("B", "C"):
    same = runs[f"{c} new"][["prediction", "n_input_tokens"]].equals(runs[f"{c} pilot"][["prediction", "n_input_tokens"]])
    print(f"{c}: " + ("the same answers and input lengths as the pilot for all 200 questions" if same
                      else "differs from the pilot (answers or input lengths)"))

table = pd.DataFrame({c: summarize(df.reset_index()) for c, df in runs.items()}).round(1)
table["B new − pilot"] = (table["B new"] - table["B pilot"]).round(1)
table["C new − pilot"] = (table["C new"] - table["C pilot"]).round(1)
table["C − B (new)"] = (table["C new"] - table["B new"]).round(1)
print(table.to_string())
for c in ("B", "C"):
    old, new = runs[f"{c} pilot"], runs[f"{c} new"]
    changed = (old["prediction"] != new["prediction"])
    by_task = new.loc[changed, "question_type"].value_counts().to_dict()
    print(f"\n{c}: {changed.sum()} of {len(IDS)} answers differ from the pilot's; by type {by_task}")
    print(f"{c}: input tokens, median {int(new['n_input_tokens'].median())} (pilot {int(old['n_input_tokens'].median())}); "
          f"replies cut off by the token limit: {int(new['hit_token_limit'].sum())}")
table.to_csv(f"{RESULTS_DIR}/preview_by_task.csv")
print(f"\nsaved {RESULTS_DIR}/preview_by_task.csv")